In [1]:
# I put this in it own block cos it keeping error, maybe because it have to finish runningt this before calling other library
!pip install datasets gensim nltk scikit-learn wefe pandas tqdm

In [2]:
# Task 1-CBOW and Skip-gram
# use dataset loading code from the homework2file
from datasets import load_dataset

import string
import nltk
from nltk.tokenize import word_tokenize
from gensim.models import Word2Vec
import logging
from tqdm import tqdm # I try this since it seems to think so long

# tracks Gensim's internal code.
logging.basicConfig(format='%(asctime)s : %(levelname)s : %(message)s', level=logging.INFO)

#I load 'punkt_tab' too, I think it fix someting,when using only 'punkt',it was not worked
nltk.download('punkt')
nltk.download('punkt_tab')


# this one from homework file
dataset = load_dataset("wikimedia/wikipedia", "20231101.simple", split="train")

print("Check the characters of Wikipedia ---")
print(dataset[0]["text"][:200])


# applying text normalization.
def clean_and_tokenize(text):
    text = text.lower()
    #i try this, it is faster than using a `for` loop or `.replace()` i usually use
    text = text.translate(str.maketrans('', '', string.punctuation))
    return word_tokenize(text)


#added tqdm() to know the exact estimated time
sentences = [clean_and_tokenize(doc) for doc in tqdm(dataset['text'][:10000], desc="Tokenizing Articles")]

#trainig process
print("Continuous Bag-of-Words (CBOW)")
# increased workers to 8 to train faster here
model_cbow = Word2Vec(sentences=sentences, vector_size=100, window=5, min_count=5, sg=0, workers=8)

print("Skip-gram")
model_skipgram = Word2Vec(sentences=sentences, vector_size=100, window=5, min_count=5, sg=1, workers=8)

print("Saving models locally")
model_cbow.save("simple_wiki_cbow.model")
model_skipgram.save("simple_wiki_skipgram.model")




[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Check the characters of Wikipedia ---
April (Apr.) is the fourth month of the year in the Julian and Gregorian calendars, and comes between March and May. It is one of the four months to have 30 days.

April always begins on the same day 


Tokenizing Articles: 100%|██████████| 10000/10000 [00:25<00:00, 384.83it/s]


Continuous Bag-of-Words (CBOW)
Skip-gram
Saving models locally


In [3]:
# Task2 - load legacy models and run queries.
# I adapt the code from class notebook (NLP_W26_embeddings.ipynb)

import gensim.downloader as api

print("*****The legacy model*****")

#  first legacy - downloading Google News 300, i use this from/as the primary example in notbook class, i think it is the industry standard too.
#  second legacy models use GloVe and select 100 as
wv_google = api.load('word2vec-google-news-300')
wv_glove = api.load('glove-wiki-gigaword-100')

# store all four models so we can loop through while using queries
models = {
    "My CBOW": model_cbow.wv,
    "My Skip-gram": model_skipgram.wv,
    "My Google News (Legacy)": wv_google,
    "My GloVe (Legacy)": wv_glove
}

print("Running 5 semantic queries")
#  use that syntax from class for 5 queries.



five_queries = [
    # 1: Family/Gender analogy (replacing King/Queen)
    {"name": "Analogy: Brother - Man + Woman", "pos": ['brother', 'woman'], "neg": ['man']},

    # 2: Geography analogy (replacing Paris/France)
    {"name": "Analogy: Rome - Italy + Japan", "pos": ['rome', 'japan'], "neg": ['italy']},

    # 3: Grammatical tense analogy (replacing Walk/Walked)
    {"name": "Analogy: Swam - Swim + Run", "pos": ['swam', 'run'], "neg": ['swim']},

    # 4: Pure similarity (replacing piano)
    {"name": "Similarity: 10 closest to 'guitar'", "pos": ['guitar'], "neg": []},

    # 5: Similarity vs Relatedness (replacing coffee)
    {"name": "Similarity: 10 closest to 'tea'", "pos": ['tea'], "neg": []}
]

"""

five_queries = [
    # 1: Classic gender analogy (from notebook in class)
    {"name": "Analogy: King - Man + Woman", "pos": ['king', 'woman'], "neg": ['man']},

    # 2: Geography analogy (i add this)
    {"name": "Analogy: Paris - France + Germany", "pos": ['paris', 'germany'], "neg": ['france']},

    # 3: Grammatical tense analogy (i add this)
    {"name": "Analogy: Walked - Walk + Run", "pos": ['walked', 'run'], "neg": ['walk']},

    # 4: Pure similarity (from homework file)
    {"name": "Similarity: 10 closest to 'piano'", "pos": ['piano'], "neg": []},

    # 5: Similarity vs Relatedness (from homework file)
    {"name": "Similarity: 10 closest to 'coffee'", "pos": ['coffee'], "neg": []}
]

"""

# run the 5 queries and compare the outputs!
# run all 4 models thru varaible models

for query in five_queries:
    print(f"\n-- {query['name']}")
    for model_name, wv in models.items():
        try:
            # use top n=3 for analogies, and top n=10 for pure similarity

            top_n = 10 if not query['neg'] else 3
            result = wv.most_similar(positive=query['pos'], negative=query['neg'], topn=top_n)


            words_only = [res[0] for res in result]
            print(f"{model_name:20}: {words_only}")


# Note: AI suggested me to add a try-except block to handle KeyErrors "" Error is " NameError: name 'queries' is not defined"""
# Because I'm using a loop not a single lines, this section prevents the script from crashing if a query word isn't found in the smaller vocabulary of my models
        except KeyError as e:
            # handles the case that small Wikipedia dataset might not have learned a specific word that Google News knows.
            print(f"{model_name:20}: Word missing from vocabulary ({e})")



*****The legacy model*****
Running 5 semantic queries

-- Analogy: Brother - Man + Woman
My CBOW             : ['cousin', 'daughter', 'wife']
My Skip-gram        : ['wife', 'husband', 'halfsister']
My Google News (Legacy): ['sister', 'daughter', 'mother']
My GloVe (Legacy)   : ['daughter', 'wife', 'mother']

-- Analogy: Rome - Italy + Japan
My CBOW             : ['greeks', 'caesar', 'china']
My Skip-gram        : ['emperors', 'japanese', 'byzantium']
My Google News (Legacy): ['japanese', 'korean', 'tokyo']
My GloVe (Legacy)   : ['tokyo', 'beijing', 'seoul']

-- Analogy: Swam - Swim + Run
My CBOW             : ['sued', 'charted', 'quit']
My Skip-gram        : ['exceeded', 'hewlettpackard', '10year']
My Google News (Legacy): ['ran', 'runs', 'drove']
My GloVe (Legacy)   : ['ran', 'went', 'drove']

-- Similarity: 10 closest to 'guitar'
My CBOW             : ['bass', 'drums', 'percussion', 'vocals', 'guitars', 'saxophone', 'harmonica', 'keyboard', 'rhythm', 'keyboards']
My Skip-gram        

In [4]:
# Task3 - Bias in word embeddings(WEFE framework)
# apply bug-fix snippet from homework file  to ensure it runs

from wefe.query import Query
from wefe.word_embedding_model import WordEmbeddingModel
from wefe.metrics import WEAT
from wefe.utils import run_queries

# Wrap 4 existing models into WEFE WordEmbeddingModel objects
wefe_models = [
    WordEmbeddingModel(model_cbow.wv, 'My CBOW'),
    WordEmbeddingModel(model_skipgram.wv, 'My Skip-gram'),
    WordEmbeddingModel(wv_google, 'MY Google News'),
    WordEmbeddingModel(wv_glove, 'My GloVe')
]


# define a new bias test for Age vs. Technology
# to prove a bias exists, I use a nature word as a contrast
# I refined word sets to increase experimental definitive results. (originally i used only 5 words in each groups)
print("Running custom WEAT Extension: Age vs. Technology Bias")
#youth_words = ['child', 'teen', 'youth', 'young', 'student']
#elderly_words = ['old', 'elderly', 'senior', 'grandfather', 'grandmother']

#tech_words = ['computer', 'internet', 'technology', 'phone', 'digital']
#nature_words = ['garden', 'farm', 'nature', 'tree', 'traditional']

youth_words = ['child', 'teen', 'youth', 'young', 'student', 'teenager', 'adolescent', 'boy', 'girl','children']
elderly_words = ['old', 'elderly', 'senior', 'grandfather', 'grandmother', 'retired', 'pensioner', 'ancestor', 'aged', 'mature']

tech_words = ['computer', 'internet', 'technology', 'phone', 'digital', 'software', 'website', 'online', 'data', 'electronic', 'coding','ai']
nature_words = ['garden', 'farm', 'nature', 'tree', 'traditional', 'forest', 'field', 'wild', 'rural', 'natural']



# create the WEAT Query object
age_tech_query = Query(
    [youth_words, elderly_words],
    [tech_words, nature_words],
    ['Youth', 'Elderly'],
    ['Technology', 'Nature']
)

# run the query using the bug-fix code from homework file
wefe_results = run_queries(
    WEAT,
    [age_tech_query],
    wefe_models,

    metric_params={
        'preprocessors': [{}, {}, {'lowercase': True}]
    }
).T.round(2)

print("\nWEAT Bias Results (Positive score = Youth/Tech bias, Negative = Elderly/Tech bias):")
print(wefe_results)

Running custom WEAT Extension: Age vs. Technology Bias

WEAT Bias Results (Positive score = Youth/Tech bias, Negative = Elderly/Tech bias):
model_name                                   My CBOW  My Skip-gram  \
query_name                                                           
Youth and Elderly wrt Technology and Nature     0.78          0.29   

model_name                                   MY Google News  My GloVe  
query_name                                                             
Youth and Elderly wrt Technology and Nature            0.16      0.46  


In [5]:
# Task4- Text classification(SPARSE VS DENSE)
# I use the Logistic Regression setup from class by swap the 'IMDB'dataset for 'rotten_tomatoes'

from datasets import load_dataset
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import numpy as np

print("Loading dataset for Sentiment Analysis")

dataset_cls = load_dataset("rotten_tomatoes")

# set the Sanity check
print("--- Sample Review and Label ---")
print("Review:", dataset_cls['train']['text'][0])
print("Label:", dataset_cls['train']['label'][0])


train_texts = dataset_cls['train']['text']
train_labels = dataset_cls['train']['label']
test_texts = dataset_cls['test']['text']
test_labels = dataset_cls['test']['label']

print("\nTraining Model SPARSE")
#limit to 10,000 features to simulate a standard sparse representation
vectorizer = CountVectorizer(max_features=10000)
X_train_sparse = vectorizer.fit_transform(train_texts)
X_test_sparse = vectorizer.transform(test_texts)

#builds a classifier (clf_sparse) and trains it using your massive 10,000-column Bag of Words data (X_train_sparse).

clf_sparse = LogisticRegression(max_iter=1000)
clf_sparse.fit(X_train_sparse, train_labels)
acc_sparse = accuracy_score(test_labels, clf_sparse.predict(X_test_sparse))

print(f"Sparse Model Accuracy: {acc_sparse:.4f}")
print(f"Sparse Feature Dimensionality: {X_train_sparse.shape[1]} features")

# build a new function to average the embeddings from Task1  and feed those averages into the exact same Logistic Regression model.
print("\nTraining Model Dense")

def document_vector(text, wv):

    # originally clean and tokenize the text using function from Task1 ( the line below ) and it doesnt work so fix it by using tokens = text.lower().split()
    # tokens = clean_and_tokenize(text)
    # using a memory-safe string split to prevent the silent crash from out of the memory
    tokens = text.lower().split()

    # get the vector for each word if it exists in model's vocab
    vectors = [wv[word] for word in tokens if word in wv]

    # return a vector of zeros, if no words are found
    if not vectors:
        return np.zeros(wv.vector_size)

    # average the vectors together
    return np.mean(vectors, axis=0)

# convert all reviews into averaged vectors using our custom CBOW model from Task 1
X_train_dense = np.array([document_vector(t, model_cbow.wv) for t in train_texts])
X_test_dense = np.array([document_vector(t, model_cbow.wv) for t in test_texts])

# fit the dense classifier
clf_dense = LogisticRegression(max_iter=1000)
clf_dense.fit(X_train_dense, train_labels)
acc_dense = accuracy_score(test_labels, clf_dense.predict(X_test_dense))

print(f"Dense Model Accuracy: {acc_dense:.4f}")
print(f"Dense Feature Dimensionality: {X_train_dense.shape[1]} features")

Loading dataset for Sentiment Analysis
--- Sample Review and Label ---
Review: the rock is destined to be the 21st century's new " conan " and that he's going to make a splash even greater than arnold schwarzenegger , jean-claud van damme or steven segal .
Label: 1

Training Model SPARSE
Sparse Model Accuracy: 0.7702
Sparse Feature Dimensionality: 10000 features

Training Model Dense
Dense Model Accuracy: 0.6295
Dense Feature Dimensionality: 100 features
